In [11]:
import numpy as np
import torch
from scipy.linalg import eigh
import networkx as nx

# PPNE

In [150]:
def MF_deep_walk(A, walk_length, negative_samples_size, embedding_dim):
    vol_A = torch.sum(A)
    D_inversed = torch.inverse(torch.diag(torch.sum(A, dim=1)))
    sum_matrix = torch.zeros(A.shape).double()
    for r in range(walk_length):
        sum_matrix += torch.linalg.matrix_power(D_inversed @ A, r+1) @ D_inversed

    Z = torch.log(torch.maximum((vol_A/(walk_length*negative_samples_size)) * sum_matrix, torch.tensor(1)))
    
    U, S, Vh = torch.linalg.svd(Z, full_matrices=False)

    X = U[:, :embedding_dim] @ torch.sqrt(torch.diag(S[:embedding_dim]))
    Y = (torch.sqrt(torch.diag(S[:embedding_dim])) @ Vh[:embedding_dim,:]).T 
    return X, Y, Z

In [ ]:
def calculate_grad_Z(X, Y, Z, X_grad):
    n, d = X.shape
    Omega = (Z != 0.0)
    grad_Z = torch.zeros((n,n)).double()
    for i in range(n):
        if torch.nonzero(X_grad[i]).numel() == 0: 
            # if the ith node is not used to calculate privacy loss, its embeddings has 0 gradients
            # this if statement is to avoid calculating the gradient of Z for this node and speed up the computation
            continue

        # calculating grad_Z, a matrix of size n x n containing 
        # the gradient of each Zij w.r.t. each Xij of the ith row of X
        # the rows are for the Xij, the cols are for the Zij
        Omega_i = np.argwhere(Omega[i])[0]
        y_yt_sum = torch.zeros((d,d)).double()
    
        for j in Omega_i:
            y = Y[j].reshape(1,-1)
            y_yt_sum += torch.mm(torch.t(y),y)

        y_yt_sum_inv = torch.linalg.inv(y_yt_sum)
        
        grad_Z[i] = torch.mm(X_grad[i].reshape(1,-1), torch.mm(y_yt_sum_inv, Y.t()))[0]
        grad_Z[i, np.argwhere(~Omega[i]).T[0]] = 0

    return grad_Z

In [152]:
def privacy_loss(X, pos_edges, neg_edges):
    cosine_similarities = X @ X.T

    i_s, j_s = np.array(pos_edges).T
    pos_similarities = torch.sum(cosine_similarities[i_s, j_s])

    i_s, j_s = np.array(neg_edges).T
    neg_similarities = torch.sum(cosine_similarities[i_s, j_s])

    return pos_similarities - neg_similarities

In [153]:
def proxy_utility_loss(A, delta, walk_length, negative_samples_size, embedding_dim):
    
    # calculating sigma_p
    degrees = np.sum(A, axis=1)
    D = np.diag(degrees)
    generalized_eigh_values, _ = eigh(A, D)
    generalized_eigh_values = generalized_eigh_values[0:A.shape[0]-embedding_dim] # get |V| - K lowest eigenvalues 

    eigenvalue_power_sum = np.zeros(len(generalized_eigh_values))
    for r in range(walk_length):
        eigenvalue_power_sum += np.power(generalized_eigh_values, r+1)
    
    
    d_min = np.min(degrees)
    sigma_p = (1/d_min) * np.abs(eigenvalue_power_sum)

    # calculating the final outputs
    vol_A = np.sum(A)
    output = ((vol_A + 2*delta)/(walk_length*negative_samples_size)) * np.sqrt(np.sum(np.power(sigma_p, 2)))
    
    return output

In [ ]:
def PPNE(A, sampling_size, batching_size, n_iterations, walk_length, negative_samples_size, embedding_dim, pos_edges, neg_edges, k, verbose=False, eval=False):
    
    A_perturbed = A.copy()
    delta = (A_perturbed == 0) - A_perturbed # change in 1 edge to create candidate perturbations
    current_privacy_gain = 0
    for ITER in range(n_iterations):

        print(f"Iteration {ITER+1}/{n_iterations}")
        ### Evaluating and Logging ###
        if eval:
            # TODO: Implement Evaluation
            pass

        ### Opimizing ###
        ### Compute Privacy Gains for all pertubations - using torch
        A_perturbed_torch = torch.tensor(A_perturbed, requires_grad=True).double()
        X, Y, Z = MF_deep_walk(A_perturbed_torch, walk_length, negative_samples_size, embedding_dim)        

        # Calculate X_grad
        print("Computing X_grad")
        X_detached = X.detach().requires_grad_(True)
        pl = privacy_loss(X_detached, pos_edges, neg_edges)
        pl.backward()

        # Calculate Z_grad
        print("Computing Z_grad")
        with torch.no_grad():
            Z_grad = calculate_grad_Z(X, Y, Z, X_detached.grad)

        # Calculate A_grad
        print("Computing A_grad")
        loss = (Z_grad * Z).sum()
        loss.backward()

        # Calculate privacy gain and clean up
        perturbation_privacy_gains = current_privacy_gain - A_perturbed_torch.grad.numpy() * delta
        X_detached.grad.zero_()
        A_perturbed_torch.grad.zero_()

        ### Compute Utility Loss for all pertubations - using numpy
        print("Computing Utility Loss")
        perturbation_utility_losses = proxy_utility_loss(A_perturbed, delta, walk_length, negative_samples_size, embedding_dim)

        ### Get the best perturbation
        scores = perturbation_privacy_gains / (perturbation_utility_losses + 1)**k
        scores = scores + np.eye(scores.shape[0]) * -1e10 # exclude self edges
        idx = np.argmax(scores)
        row = idx // scores.shape[0]
        col = idx % scores.shape[0]

        ### Update the best perturbation
        if verbose:
            if delta[row, col] == 1:
                print(f"Adding edge ({row}, {col})")
            else:
                print(f"Removing edge ({row}, {col})")

        A_perturbed[row, col] = A_perturbed[row, col] + delta[row, col]
        A_perturbed[col, row] = A_perturbed[col, row] + delta[col, row]
        delta[row, col] = delta[row, col] * -1
        delta[col, row] = delta[row, col] * -1
        current_privacy_gain = perturbation_privacy_gains[row,col]
    
    with torch.no_grad():
        X, _, _ = MF_deep_walk(torch.tensor(A_perturbed), walk_length, negative_samples_size, embedding_dim)
        
    return X.numpy(), A_perturbed

# Processing data

In [17]:
def sample_private_nodes(adj_matrix, private_ratio=0.1):

    # sample a number of private nodes
    num_nodes = adj_matrix.shape[0]

    num_private_nodes = int(num_nodes * private_ratio)

    while True:
        g_copy = nx.Graph(adj_matrix)
        private_nodes = np.random.choice(num_nodes, num_private_nodes, replace=False)
        private_edges = list(g_copy.subgraph(private_nodes).edges())
        # remove private edges
        g_copy.remove_edges_from(private_edges)
        # if isolated nodes exist, remove them
        isolate_nodes = list(nx.isolates(g_copy))

        if len(isolate_nodes) == 0:
            print('no isolated nodes')
            private_edges = np.array(private_edges)
            num_private_edges = private_edges.shape[0]
            private_non_edges = []
            G_private = nx.Graph()
            G_private.add_edges_from(private_edges)
            while len(private_non_edges) < num_private_edges:
                u, v = np.random.choice(private_nodes, 2, replace=False)
                if G_private.has_edge(u, v):
                    continue
                elif [u, v] in private_non_edges:
                    continue
                elif [v, u] in private_non_edges:
                    continue
                else:
                    private_non_edges.append([u, v])
            adj_matrix = nx.to_numpy_array(g_copy) # edited
            private_edges = np.array(private_edges)
            private_non_edges = np.array(private_non_edges)
            break
        else:
            print("isolated nodes exist. re-sample.")
            continue

    return adj_matrix, private_edges, private_non_edges


In [18]:
G = nx.read_edgelist("./datasets/cora/cora.cites", nodetype=int)

In [19]:
np.random.seed(1)
A, pos_edges, neg_edges = sample_private_nodes(nx.to_numpy_array(G))

isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
no isolated nodes


In [155]:
perturbed_embedding, perturbed_adj = PPNE(A, sampling_size=10000, batching_size=1, n_iterations=1, 
                                          walk_length=5, negative_samples_size=10, embedding_dim=10, 
                                          pos_edges=pos_edges, neg_edges=neg_edges, k=1, verbose=True, eval=True)


Iteration 1/1
Computing X_grad
Computing Z_grad
Computing A_grad
Computing Utility Loss
Adding edge (101, 124)
